# Praktikum 4: Language Modeling with RNN & LSTM
In these exercises, you will systematically explore how model capacity, sequence length, and dataset size influence the performance of RNNs and LSTMs on language modeling.


## 1. Setup and Base Code

### 1.1 Preparing the corpus
We use the The Penn Treebank dataset, a corpus that has been cleaned for language modeling tasks.
 If you don't have access to internet from your notebook, use the script `download_dataset.py` and then `load_from_disk()`

To download from the terminal:
````bash
$ python scripts/download_dataset.py "ptb-text-only/ptb_text_only" data/ptb_text_only
````


In [ ]:
from datasets import load_dataset, load_from_disk

ds = load_from_disk("../../data/ptb_text_only")

In [ ]:
import pandas as pd

# Convert splits to pandas DataFrames
df_train_full = ds["train"].to_pandas()
df_test_full = ds["test"].to_pandas()

print("Train shape:", df_train_full.shape)
print("Test  shape:", df_test_full.shape)

df_train_full.head()

### 1.2 Sample and prepare the splits
We prepare a helper function to more easily sample a given number of texts documents from the dataset: `sample_dataset()`.  

In [ ]:
import random

NUM_TRAIN = 20_000
NUM_TEST =  3_000

def sample_dataset(df_train, df_test, num_train, num_test):
    train_texts = df_train["sentence"].to_list()
    eval_texts = df_test["sentence"].to_list()
    
    train_sm  = random.sample(train_texts,  num_train)
    eval_sm   = random.sample(eval_texts, num_test)
    return train_sm, eval_sm

train_text_sm, eval_text_sm = sample_dataset(df_train_full, df_test_full, NUM_TRAIN, NUM_TEST)

And below we functions to tokenize a sample of documents: `build_tokenized_docs()`.

In [ ]:
import spacy
from spacy.symbols import ORTH

nlp = spacy.blank("en")

# Tell spaCy that "<unk>" is a single token
special_case = [{ORTH: "<unk>"}]
nlp.tokenizer.add_special_case("<unk>", special_case)

def tokenize(text):
    doc = nlp.make_doc(text)
    tokens = []
    for tok in doc:
        if tok.text == "<unk>":          # our special token
            tokens.append("<unk>")
        elif tok.is_alpha:
            tokens.append(tok.text.lower())
        elif tok.text in [".", "!", "?"]:
            tokens.append(tok.text)
    return ["<s>"] + tokens + ["</s>"]

def build_tokenized_docs(train_text, test_text):
    # Build tokenised docs
    train_docs = [tokenize(t) for t in train_text]
    test_docs  = [tokenize(t) for t in test_text]
    return train_docs, test_docs

train_docs, test_docs = build_tokenized_docs(train_text_sm, eval_text_sm)

list(tokenize("Pierre <unk> N years"))


### 1.3 Building the vocabulary
We build a **vocabulary** of all words in the training set, as well as an index that will allow us to encode sentences as an index. We provide a helper function to easily build / rebuild the vocabulary from a set of documents, considering those tokens appearing at least `min_freq`.

In [ ]:
from collections import Counter

# keep only frequent words to limit vocab size
MIN_FREQ = 3

def build_vocabulary(train_docs, min_freq):
    counter = Counter(tok for s in train_docs for tok in s)
    
    #vocab = ["<unk>"] + [w for w,c in counter.items() if c >= min_freq] # <unk> is already in the vocab!
    vocab = [w for w,c in counter.items() if c >= min_freq]
    word2idx = {w:i for i,w in enumerate(vocab)}
    idx2word = {i:w for w,i in word2idx.items()}
    return vocab, word2idx, idx2word

vocab, word2idx, idx2word = build_vocabulary(train_docs, MIN_FREQ)

len(vocab)

### 1.4 Inspect and summarize docs properties
Below we provide some details about the dataset. These are important to anticipate how well the models might work.

In [ ]:
from collections import Counter
import numpy as np

print("=== Docs & length stats ===")
print(f"# train docs: {len(train_docs)}")
print(f"# test docs : {len(test_docs)}")

train_lens = [len(d) for d in train_docs]
test_lens  = [len(d) for d in test_docs]

def summarize_lens(name, lens):
    print(f"{name} len (tokens) → min: {min(lens)}, max: {max(lens)}, mean: {np.mean(lens):.2f}")

summarize_lens("Train", train_lens)
summarize_lens("Test ", test_lens)

# Type / vocab overlap
train_types = set(tok for doc in train_docs for tok in doc)
test_types  = set(tok for doc in test_docs for tok in doc)

overlap = train_types & test_types
only_test = test_types - train_types

print("\n=== Type / vocab overlap ===")
print(f"Train types        : {len(train_types)}")
print(f"Test types         : {len(test_types)}")
print(f"Overlap types      : {len(overlap)}")
print(f"Types only in test : {len(only_test)}")

# Show a few tokens that appear only in test (sanity check)
print(f"Sample 'only in test' tokens: {list(sorted(only_test))[:20]}")


### 1.5 Encoding documents
Using our vocabulary, we encode documents as array of indexes. We provide a helper `encode_data_split()` to encode tokenized documents.

In [ ]:
import numpy as np
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
import torch

UNK = word2idx["<unk>"]

def encode_doc(tokens):
    """Map a tokenised review to a list of token IDs."""
    return [word2idx.get(tok, UNK) for tok in tokens]

def encode_data_split(split_docs):
    encoded_docs = [
        torch.tensor(encode_doc(doc), dtype=torch.long)
        for doc in split_docs
        if len(doc) >= 2  # is more than just <s></s>
    ]
    return encoded_docs

    
# Encode all training and test reviews
encoded_train_docs = encode_data_split(train_docs)
encoded_test_docs = encode_data_split(test_docs)

len(encoded_train_docs), len(encoded_test_docs)


### 1.6 Define models

In [ ]:
import torch.nn as nn

class TinyRNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        emb = self.embed(x)
        out, h = self.rnn(emb, h)
        logits = self.fc(out)
        return logits, h

In [ ]:
import torch.nn as nn

class TinyLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, h=None):
        emb = self.embed(x)               # (batch, seq_len, embed_dim)
        out, h = self.lstm(emb, h)        # out: (batch, seq_len, hidden_dim)
                                          # h: (h_T, c_T)
        logits = self.fc(out)             # (batch, seq_len, vocab_size)
        return logits, h

### 1.7 Training loop

In [ ]:
def train_lm(
    model,
    encoded_sents,
    optimizer,
    criterion,
    device,
    epochs=10,
    eval_sents=None,
    min_delta=1e-3,
    return_losses=False
):
    """
    Train a language model (RNN or LSTM) on encoded sentences.

    - Processes ONE sentence/document at a time.
    - Tracks training loss.
    - Optionally tracks validation loss.
    - Prints loss change so students can judge convergence.
    """

    prev_train_loss = None
    prev_val_loss = None
    
    train_losses = []
    val_losses = []

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        used_sents = 0

        random.shuffle(encoded_sents)

        for ids in encoded_sents:
            if ids.size(0) < 2:
                continue

            ids = ids.to(device)
            x = ids[:-1].unsqueeze(0)
            y = ids[1:]

            optimizer.zero_grad()

            logits, _ = model(x)
            logits = logits.squeeze(0)

            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            used_sents += 1

        avg_train_loss = total_loss / max(used_sents, 1)
        train_losses.append(avg_train_loss)

        # Training delta
        if prev_train_loss is None:
            train_delta = 0.0
        else:
            train_delta = prev_train_loss - avg_train_loss
        
        prev_train_loss = avg_train_loss
        
        # Validation loss + delta
        if eval_sents is not None:
            avg_val_loss = evaluate_lm_loss(
                model, eval_sents, criterion, device
            )
            val_losses.append(avg_val_loss)

            if prev_val_loss is None:
                val_delta = 0.0
            else:
                val_delta = prev_val_loss - avg_val_loss
        
            prev_val_loss = avg_val_loss

            print(
                f"Epoch {epoch+1:02d}: "
                f"train={avg_train_loss:.4f} "
                f"(Δ={train_delta:+.4f}) | "
                f"val={avg_val_loss:.4f} "
                f"(Δ={val_delta:+.4f})"
            )
        else:
            print(
                f"Epoch {epoch+1}: "
                f"avg loss = {avg_train_loss:.4f} | "
                f"Δ = {delta:+.4f}"
            )

        if train_delta > 0 and train_delta < min_delta:
            print("Small improvement detected — stopping early.")
            break

    if return_losses:
        if eval_sents is not None:
            return train_losses, val_losses
        return train_losses

In [ ]:
def evaluate_lm_loss(model, encoded_sents, criterion, device):
    """
    Evaluate a language model sentence by sentence.
    No batching, no padding.
    """
    model.eval()
    total_loss = 0.0
    used_sents = 0

    with torch.no_grad():
        for ids in encoded_sents:
            if ids.size(0) < 2:
                continue

            ids = ids.to(device)
            x = ids[:-1].unsqueeze(0)   # (1, seq_len)
            y = ids[1:]                 # (seq_len,)

            logits, _ = model(x)
            logits = logits.squeeze(0)  # (seq_len, vocab_size)

            loss = criterion(logits, y)

            total_loss += loss.item()
            used_sents += 1

    return total_loss / max(used_sents, 1)

### 1.8 Text generation
We provide a helper function `generate_lm()` that can take one of the language models, and generate a document. 

In [ ]:
import math
import torch.nn.functional as F

SOS = word2idx["<s>"]  # index of the start of the sentence
EOS = word2idx["</s>"] # index of the end of the sentence


def generate_lm(model, max_len=25):
    """
    Generate a sentence from the trained language model (RNN / LSTM).

    - Starts from <s>
    - At each step: predict next-word distribution and sample one token
    - Stops when </s> is produced or max_len is reached
    """
    model.eval()
    h = None              # initial hidden state (RNN will start with zeros)
    idx = SOS             # start from <s>
    generated = []        # list of token strings

    for _ in range(max_len):
        # current input is a single token ID, shaped as (batch=1, seq_len=1)
        x = torch.tensor([[idx]], dtype=torch.long, device=device)

        with torch.no_grad():
            # forward one step: use previous hidden state h
            logits, h = model(x, h)   # logits: (1, 1, vocab_size)

        # convert logits of the last (and only) time step to probabilities
        probs = F.softmax(logits[0, -1], dim=-1)

        # sample a token ID according to the probability distribution
        idx = torch.multinomial(probs, num_samples=1).item()

        # stop if we generated the end-of-sentence token
        if idx == EOS:
            break

        # store the generated word
        generated.append(idx2word[idx])

    # join all generated tokens into a single string
    return " ".join(generated)

### 1.9 Evaluation

To evaluate our language models, we use the same metrics introduced in the
N-gram lab: **cross-entropy** and **perplexity**.  
These metrics quantify how well a model predicts the next token in a held-out
test set.

As with training, we must process the test split using **the exact same
pipeline**: cleaning, tokenising, adding `<s>` and `</s>` markers, and encoding
into token IDs. We then run the model forward over every sentence in the test
set and accumulate the total negative log-likelihood.

The helper function below performs this procedure and returns:

- cross-entropy (nats per token),
- perplexity,
- and the number of test sentences used.



In [ ]:
criterion_eval = nn.CrossEntropyLoss(reduction="sum")  # sum over tokens

def lm_cross_entropy_perplexity_from_sents(model, sents, max_len=None, device=None):
    """
    Compute corpus cross-entropy (nats/token) and perplexity for an RNN/LSTM LM.

    sents: list of token lists, e.g. ["<s>", "in", "the", "city", "</s>"]
    max_len: optional cap on sentence length
    """
    model.eval()
    if device is None:
        device = next(model.parameters()).device

    total_loss = 0.0
    total_tokens = 0
    used_sents = 0

    with torch.no_grad():
        for toks in sents:
            if max_len is not None:
                toks = toks[:max_len]

            ids = toks

            ids_tensor = ids.to(device)
            x = ids_tensor[:-1].unsqueeze(0)   # (1, seq_len)
            y = ids_tensor[1:]                 # (seq_len,)

            logits, _ = model(x)               # (1, seq_len, vocab)
            logits = logits.squeeze(0)         # (seq_len, vocab)

            loss = criterion_eval(logits, y)
            total_loss += loss.item()
            total_tokens += y.size(0)
            used_sents += 1

    if total_tokens == 0:
        return math.inf, math.inf, 0

    H = total_loss / total_tokens
    PP = math.exp(H)
    return H, PP, used_sents


## 2. Tasks and exploration
Below we provide a helper function to easily run the experiments, based on the functions previously defined. In each run we can modify model and training parameters.

In [ ]:
import matplotlib.pyplot as plt

def run_experiment(
        model_type="LSTM",
        hidden_dim=64,
        embed_dim=32,
        num_train_docs=5000,
        epochs=10,
        eval_buckets=None,
        generate=True,
        generate_len=30,
        plot_loss=False,
        verbose=True):

    # -----------------------------
    # 1. Sample training subset
    # -----------------------------
    train_subset = random.sample(encoded_train_docs, num_train_docs)

    if verbose:
        print(f"\n=== Running experiment ===")
        print(f"Model      : {model_type}")
        print(f"Hidden dim : {hidden_dim}")
        print(f"Embed dim  : {embed_dim}")
        print(f"Train docs : {num_train_docs}")
        print(f"Epochs     : {epochs}")
        print("-----------------------------")

    # -----------------------------
    # 2. Build model
    # -----------------------------
    vocab_size = len(word2idx)

    if model_type.upper() == "RNN":
        model = TinyRNN(vocab_size=vocab_size,
                        embed_dim=embed_dim,
                        hidden_dim=hidden_dim).to(device)
    else:
        model = TinyLSTM(vocab_size=vocab_size,
                         embed_dim=embed_dim,
                         hidden_dim=hidden_dim).to(device)

    if verbose:
        print(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

    # -----------------------------
    # 3. Training loop (mirrors train_lm but records loss per epoch)
    # -----------------------------
    train_losses, val_losses = train_lm(model, train_subset, optimizer, criterion, 
                                device, epochs=epochs, 
                                eval_sents=encoded_test_docs,
                                return_losses=plot_loss)

    # -----------------------------
    # 4. Optional plotting
    # -----------------------------
    if plot_loss:
        plt.figure(figsize=(6,4))
        plt.plot(train_losses, label="Training loss")
        plt.plot(val_losses, label="Validation loss")
        plt.title("Training Loss per Epoch")
        plt.xlabel("Epoch")
        plt.ylabel("Loss")
        plt.legend()
        plt.grid(True)
        plt.show()

    # -----------------------------
    # 5. Evaluation on full test set
    # -----------------------------
    H_total, PP_total, used = lm_cross_entropy_perplexity_from_sents(
        model, encoded_test_docs, max_len=None, device=device
    )

    print(f"\n=== Test set evaluation ===")
    print(f"Cross-entropy : {H_total:.4f} nats/token")
    print(f"Perplexity    : {PP_total:.2f}")
    print(f"Sentences used: {used}")

    # -----------------------------
    # 6. Bucket evaluation (if provided)
    # -----------------------------
    if eval_buckets:
        print("\n=== Evaluation by buckets ===")
        for name, sents in eval_buckets.items():
            H, PP, used = lm_cross_entropy_perplexity_from_sents(
                model, sents, max_len=None, device=device)
            print(f"{name:>10s} → H={H:.4f} | PP={PP:.2f} | docs={used}")

    # -----------------------------
    # 7. Generate text (optional)
    # -----------------------------
    generated_last = None
    if generate:
        print("\n=== Sample generation ===")
        for _ in range(3):
            g = generate_lm(model, max_len=generate_len)
            print(" →", g)
        generated_last = g

    return {
        "model": model,
        "H_total": H_total,
        "PP_total": PP_total,
        "loss_curve": train_losses,
        "generated_last": generated_last
    }


### Task 1. Effect of Model Size
We change the model size to observe the performance. In particular, we try the following model sizes:

- small model:
```
embed_dim = 16
hidden_dim = 32
```
- medium model
 ```
embed_dim = 32
hidden_dim = 64
```

- larger model
```
embed_dim = 64
hidden_dim = 128
```

- XL model (optional, might be slow)
```
embed_dim = 128
hidden_dim = 256
```

**Measure**:
- training loss curve  
- test perplexity  
- generate 20–30 tokens  

**Questions**:
1. Does bigger = better?  
2. When does overfitting appear?
3. How do generated samples differ?

In [ ]:
%%time
configs = [
    # RNN configurations
    {"model_type": "RNN",  "embed_dim": 16, "hidden_dim": 32},
    {"model_type": "RNN",  "embed_dim": 32, "hidden_dim": 64},
    {"model_type": "RNN",  "embed_dim": 64, "hidden_dim": 128},

    # LSTM configurations
    {"model_type": "LSTM", "embed_dim": 16, "hidden_dim": 32},
    {"model_type": "LSTM", "embed_dim": 32, "hidden_dim": 64},
    {"model_type": "LSTM", "embed_dim": 64, "hidden_dim": 128},
]

results = []

for cfg in configs:
    print("\n======================================")
    print(f"Running: {cfg['model_type']} "
          f"(embed={cfg['embed_dim']}, hidden={cfg['hidden_dim']})")
    print("======================================")

    out = run_experiment(
        model_type=cfg["model_type"],
        embed_dim=cfg["embed_dim"],
        hidden_dim=cfg["hidden_dim"],
        num_train_docs=5000,
        epochs=10,
        plot_loss=True   # or True, if you want curves
    )
    ## let's store the output so we can more easily run the next experiment
    cfg["out"] = out

### Task 2 – Length Sensitivity (Short vs Medium vs Long)
We want to test the sensitivity of the model to different lenght, but at this point by focusing on different evaluation buckets:

  * short (≤ 15 tokens)
  * medium (16–30 tokens)
  * long (> 30 tokens)


**Measure** perplexity for short, medium, long.

**Questions**:
1. Does perplexity increase with sequence length?  
2. Does the model capture long-range dependencies?  
3. Does RNN degrade faster than LSTM?

In [ ]:
short_docs  = [s for s in encoded_test_docs if len(s) <= 20]
medium_docs = [s for s in encoded_test_docs if 20 < len(s) <= 30]
long_docs   = [s for s in encoded_test_docs if len(s) > 30]

# let's sample the same number
N = min(len(short_docs), len(medium_docs), len(long_docs))

short_sample  = random.sample(short_docs,  N)
medium_sample = random.sample(medium_docs, N)
long_sample   = random.sample(long_docs,   N)

print (len(short_sample))
print (len(medium_sample))
print (len(long_sample))
len(short_docs), len(medium_docs), len(long_docs)

# 1) Prepare the evaluation buckets (already computed earlier)
eval_buckets = {
    "all"    : encoded_test_docs,
    "short"  : short_sample,
    "medium" : medium_sample,
    "long"   : long_sample,
}

model_rnn = configs[0]["out"]["model"]
model_lstm = configs[3]["out"]["model"]

for name, sents in eval_buckets.items():
    print("==== "+ name + "====")
    H, PP, used = lm_cross_entropy_perplexity_from_sents(
        model_rnn, sents, max_len=None, device=device
    )
    print(f"RNN  {name:>10s} → H={H:.4f}, PP={PP:.2f}, docs={used}")
    
    H, PP, used = lm_cross_entropy_perplexity_from_sents(
        model_lstm, sents, max_len=None, device=device
    )
    print(f"LSTM {name:>10s} → H={H:.4f}, PP={PP:.2f}, docs={used}")

### Task 3. Training Data Size
Finally, let's assess the effect of different training datasets.
- NUM_TRAIN ∈ {1_000, 5_000, 10_000, 20_000}

To make the training maneagable, let's test only the LSTM model.

- model_type = 'LSTM'
- embed_dim = 32
- hidden_dim = 64

**Measure**:
- training loss  
- test perplexity  
- sample generation  

**Questions**:
1. How much data is needed before the model becomes reasonable?  
2. When does the model start to overfit?  
3. Does text generation quality improve with more data?

In [ ]:

train_sizes = [1000, 5000, 10000, 20000]

results_task3 = []

for n in train_sizes:
    print("\n======================================")
    print(f"Training on {n} documents")
    print("======================================")

    out = run_experiment(
        model_type="LSTM",
        embed_dim=32,
        hidden_dim=64,
        num_train_docs=n,
        epochs=20,
        plot_loss=True    # optional: set to False on slow machines
    )

    results_task3.append({
        "num_train_docs": n,
        "loss": out["H_total"],
        "perplexity": out["PP_total"],
        "generated": out["generated_last"]
    })

print("\n=== Summary (Task 3) ===")
for r in results_task3:
    print(f"{r['num_train_docs']:>6d} docs → "
          f"PP = {r['perplexity']:.2f}")


## 3. Reporting

### Task 1.
1. Does bigger = better?  
2. When does overfitting appear?
3. How do generated samples differ?
   
*Asnwer the questions here*

### Task 2.
1. Does perplexity increase with sequence length?  
2. Does the model capture long-range dependencies?  
3. Does RNN degrade faster than LSTM?

*Asnwer the questions here*

### Task 3.
1. How much data is needed before the model becomes reasonable?  
2. When does the model start to overfit?  
3. Does text generation quality improve with more data?
   
*Asnwer the questions here*